# 02 — Pelatihan dan Perbandingan Model

**Kelompok 12 · Kelas B2 · Prediksi Customer Churn Telco**

Tiga model (Logistic Regression, Decision Tree, Random Forest) dilatih dengan pipeline pra-pemrosesan yang
sama, lalu dibandingkan dengan validasi silang 5-fold memakai metrik F1. Model dengan F1 validasi silang
tertinggi disimpan sebagai model inti aplikasi.

Output: `hasil_perbandingan.csv`, `cv_detail.csv`, `model_churn.pkl`, `model_meta.pkl`, `model_info.json`.
Setelah notebook ini, jalankan `03_Kalibrasi_dan_Threshold.ipynb`.

## 1. Import library

Komponen scikit-learn untuk pipeline, validasi silang, dan tiga model. `load_training_split` dan
`classification_metrics` diambil dari `training_data.py` karena juga dipakai aplikasi Streamlit dan
pengujian otomatis, sehingga semua bagian memakai data dan cara hitung yang sama.

In [1]:
import json

import joblib
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from training_data import BASE, classification_metrics, load_training_split

## 2. Memuat data latih dan uji

`load_training_split()` membaca CSV, membuang `customerID`, mengubah `TotalCharges` menjadi angka,
menghapus 11 baris kosong, membuang kolom `TotalCharges`, lalu membagi data 80:20 secara stratified
(`random_state=42`). Target `y` bernilai 1 untuk churn dan 0 untuk tidak churn. Tabel di bawah menunjukkan
proporsi churn yang sama pada data latih dan data uji.

In [2]:
X_train, X_test, y_train, y_test = load_training_split()
print(f"Data latih: {len(X_train):,} baris | Data uji: {len(X_test):,} baris | Fitur: {X_train.shape[1]}")
display(pd.DataFrame({
    "Data latih": y_train.value_counts(normalize=True),
    "Data uji": y_test.value_counts(normalize=True),
}).rename(index={0: "Tidak churn", 1: "Churn"}).round(4))

Data latih: 5,625 baris | Data uji: 1,407 baris | Fitur: 18


,Data latih,Data uji
Churn,,
Tidak churn,0.7342,0.7342
Churn,0.2658,0.2658


## 3. Fitur numerik dan kategorikal

Tiga fitur diperlakukan sebagai angka; 15 fitur lainnya adalah kategori.

In [3]:
num = ["SeniorCitizen", "tenure", "MonthlyCharges"]
cat = [col for col in X_train.columns if col not in num]
print(f"Numerik ({len(num)}): {num}")
print(f"Kategorikal ({len(cat)}): {cat}")

Numerik (3): ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Kategorikal (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 4. Pra-pemrosesan

`StandardScaler` menyamakan skala fitur angka, `OneHotEncoder` mengubah setiap kategori menjadi kolom 0/1
(`handle_unknown="ignore"` agar kategori baru tidak membuat error). Pra-pemrosesan dimasukkan ke dalam
`Pipeline`, sehingga scaler dan encoder hanya belajar dari bagian data latih pada setiap putaran validasi.

In [4]:
pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])

## 5. Validasi silang 5-fold

Data latih dibagi menjadi 5 bagian dengan proporsi churn yang sama. Setiap model dilatih 5 kali, masing-masing
diuji pada satu bagian yang tidak ikut dilatih.

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 6. Kandidat model dan parameter

Setiap model memiliki daftar parameter yang dicoba. `class_weight="balanced"` memberi bobot lebih besar pada
kelas churn yang jumlahnya lebih sedikit; cara ini dipakai sebagai pengganti resampling (tanpa SMOTE).

In [6]:
models = {
    "LogisticRegression": (
        LogisticRegression(max_iter=1000),
        {"clf__C": [0.1, 1.0, 10.0], "clf__class_weight": [None, "balanced"]},
    ),
    "DecisionTree": (
        DecisionTreeClassifier(random_state=42),
        {"clf__max_depth": [5, 10, None], "clf__min_samples_split": [2, 5],
         "clf__class_weight": [None, "balanced"]},
    ),
    "RandomForest": (
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {"clf__n_estimators": [200], "clf__max_depth": [12, None],
         "clf__min_samples_split": [2, 5],
         "clf__class_weight": ["balanced", "balanced_subsample"]},
    ),
}

## 7. GridSearchCV untuk setiap model

Untuk setiap model dicari kombinasi parameter dengan F1 validasi silang tertinggi. Dicatat juga F1 pada data
latih (untuk melihat overfitting), F1 per fold, dan metrik pada data uji dengan threshold 0,50. Model dengan
F1 validasi silang tertinggi disimpan di variabel `best`.

In [7]:
rows, cv_details = [], []
best = None
for name, (clf, grid) in models.items():
    gs = GridSearchCV(Pipeline([("pre", pre), ("clf", clf)]), grid, cv=cv,
                      scoring="f1", n_jobs=-1, return_train_score=True)
    gs.fit(X_train, y_train)
    bi = gs.best_index_
    probabilities = gs.predict_proba(X_test)[:, 1]
    row = {
        "Model": name, "BestParams": gs.best_params_,
        "CV_F1": round(float(gs.best_score_), 4),
        "CV_F1_STD": round(float(gs.cv_results_["std_test_score"][bi]), 4),
        "CV_TRAIN_F1": round(float(gs.cv_results_["mean_train_score"][bi]), 4),
        "Train_F1": round(float(f1_score(y_train, gs.predict(X_train))), 4),
        **classification_metrics(y_test, probabilities, 0.5),
    }
    rows.append(row)
    for fold in range(5):
        cv_details.append({"Model": name, "Fold": fold + 1,
                           "F1": round(float(gs.cv_results_[f"split{fold}_test_score"][bi]), 4)})
    if best is None or gs.best_score_ > best["score"]:
        best = {"score": float(gs.best_score_), "name": name,
                "pipe": gs.best_estimator_, "row": row}
    print(f"{name}: CV-F1 {row['CV_F1']:.4f} | Train F1 {row['Train_F1']:.4f} | "
          f"Test F1 {row['F1']:.4f} | parameter {gs.best_params_}")

LogisticRegression: CV-F1 0.6304 | Train F1 0.6338 | Test F1 0.6043 | parameter {'clf__C': 10.0, 'clf__class_weight': 'balanced'}


DecisionTree: CV-F1 0.6147 | Train F1 0.6433 | Test F1 0.6163 | parameter {'clf__class_weight': 'balanced', 'clf__max_depth': 5, 'clf__min_samples_split': 2}


RandomForest: CV-F1 0.6260 | Train F1 0.8217 | Test F1 0.6096 | parameter {'clf__class_weight': 'balanced_subsample', 'clf__max_depth': 12, 'clf__min_samples_split': 5, 'clf__n_estimators': 200}


## 8. Tabel perbandingan model

Diurutkan dari F1 validasi silang tertinggi dan disimpan ke `hasil_perbandingan.csv`. Selisih besar antara
`Train_F1` dan `CV_F1` menandakan overfitting.

In [8]:
comparison = pd.DataFrame(rows).sort_values("CV_F1", ascending=False)
comparison.to_csv(BASE / "hasil_perbandingan.csv", index=False)
display(comparison.drop(columns="BestParams"))

,Model,CV_F1,CV_F1_STD,CV_TRAIN_F1,Train_F1,Accuracy,Precision,Recall,F1,ROC_AUC,Brier,TN,FP,FN,TP
0,LogisticRegression,0.6304,0.0087,0.6354,0.6338,0.7264,0.4908,0.7861,0.6043,0.8321,0.1741,728,305,80,294
2,RandomForest,0.6260,0.0126,0.8342,0.8217,0.7697,0.5548,0.6765,0.6096,0.8235,0.1567,830,203,121,253
1,DecisionTree,0.6147,0.0083,0.6366,0.6433,0.7363,0.5025,0.7968,0.6163,0.8305,0.1721,738,295,76,298


## 9. F1 setiap fold

Nilai F1 kelima fold untuk parameter terbaik setiap model, disimpan ke `cv_detail.csv`. Sebaran nilai yang
sempit berarti model stabil.

In [9]:
cv_detail = pd.DataFrame(cv_details)
cv_detail.to_csv(BASE / "cv_detail.csv", index=False)
display(cv_detail.pivot(index="Model", columns="Fold", values="F1"))

Fold,1,2,3,4,5
Model,,,,,
DecisionTree,0.6099,0.6124,0.6310,0.6127,0.6077
LogisticRegression,0.6395,0.6263,0.6154,0.6347,0.6364
RandomForest,0.6326,0.6220,0.6038,0.6407,0.6311


## 10. Menyimpan model terbaik

Pipeline terbaik disimpan ke `model_churn.pkl`, daftar fitur ke `model_meta.pkl`, dan ringkasan hasilnya ke
`model_info.json`.

In [10]:
joblib.dump(best["pipe"], BASE / "model_churn.pkl", protocol=4)
joblib.dump({"num": num, "cat": cat}, BASE / "model_meta.pkl")
info = {"best_model": best["name"], "selection_metric": "CV_F1", "metrics": best["row"],
        "n_train": len(X_train), "n_test": len(X_test)}
(BASE / "model_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
print(f"Model inti: {best['name']} | CV-F1: {best['score']:.4f}")

Model inti: LogisticRegression | CV-F1: 0.6304


## 11. Uji tambahan: model dengan dan tanpa TotalCharges

`TotalCharges` dibuang dari fitur karena isinya hampir sama dengan `tenure × MonthlyCharges`. Sel ini
membuktikannya pada data latih saja: (1) korelasi `TotalCharges` dengan `tenure × MonthlyCharges`,
(2) selisih median keduanya, dan (3) F1 validasi silang Logistic Regression terbaik dengan dan tanpa
`TotalCharges` memakai pembagian fold yang sama. Hasilnya disimpan ke `uji_totalcharges.json`.
Lanjutkan ke notebook `03_Kalibrasi_dan_Threshold.ipynb`.

In [11]:
raw = pd.read_csv(BASE / "Telco-Customer-Churn.csv")
total_charges = pd.to_numeric(raw["TotalCharges"], errors="coerce").dropna().reset_index(drop=True)
X_train_total = X_train.assign(TotalCharges=total_charges.loc[X_train.index])
product = X_train_total["tenure"] * X_train_total["MonthlyCharges"]

with_total = clone(best["pipe"]).set_params(pre__transformers=[
    ("num", StandardScaler(), num + ["TotalCharges"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])
scores = {
    "tanpa_TotalCharges": cross_val_score(clone(best["pipe"]), X_train, y_train, cv=cv, scoring="f1"),
    "dengan_TotalCharges": cross_val_score(with_total, X_train_total, y_train, cv=cv, scoring="f1"),
}
check = {
    "model": best["name"],
    "korelasi_TotalCharges_vs_tenure_x_MonthlyCharges": round(float(X_train_total["TotalCharges"].corr(product)), 4),
    "median_selisih_relatif": round(float(((X_train_total["TotalCharges"] - product).abs()
                                           / X_train_total["TotalCharges"]).median()), 4),
    "cv_f1": {name: {"mean": round(float(s.mean()), 4), "std": round(float(s.std()), 4),
                     "per_fold": [round(float(v), 4) for v in s]} for name, s in scores.items()},
    "n_train": len(X_train),
}
(BASE / "uji_totalcharges.json").write_text(json.dumps(check, indent=2), encoding="utf-8")
display(pd.DataFrame({name: v for name, v in check["cv_f1"].items()}).T[["mean", "std"]])
print(f"Korelasi TotalCharges vs tenure x MonthlyCharges: {check['korelasi_TotalCharges_vs_tenure_x_MonthlyCharges']}")
print(f"Median selisih relatif: {check['median_selisih_relatif']:.2%}")

,mean,std
tanpa_TotalCharges,0.6304,0.0087
dengan_TotalCharges,0.6302,0.0067


Korelasi TotalCharges vs tenure x MonthlyCharges: 0.9996
Median selisih relatif: 1.98%
